# 11.5 — Biodiversity and cross-analysis robustness

This final notebook reuses compatible artifacts from Notebooks 11.1–11.4, trains only missing biodiversity-assumption cases, and writes qualified descriptive screening diagnostics. These diagnostics are not statistical confirmation. Incomplete comparison groups are reported and excluded. The `test` profile uses synthetic cells; `screen` and `full` use the preserved historical feature table.

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

from estonia_landuse.sensitivity.config import DEFAULT_SEEDS
from estonia_landuse.sensitivity.historical_model import SCENARIO_LABELS
from estonia_landuse.sensitivity.robustness import (
    build_robustness_report, current_artifact_identity,
    full_manifest_for_partial_resume, inventory_artifacts,
    missing_manifest_rows,
)
from estonia_landuse.sensitivity.runner import run_manifest
from estonia_landuse.sensitivity.sampling import (
    build_biodiversity_manifest, manifest_run_count, manifest_summary,
)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
HISTORICAL_ROOT = (
    PROJECT_ROOT.parent.parent
    if PROJECT_ROOT.parent.name == ".worktrees"
    else PROJECT_ROOT
)
# PROFILE = os.environ.get("SENSITIVITY_PROFILE", "test")
PROFILE = os.environ.get("SENSITIVITY_PROFILE", "full")
N_WORKERS = int(os.environ.get("SENSITIVITY_N_WORKERS", "12"))
OVERWRITE = os.environ.get("SENSITIVITY_OVERWRITE", "false").lower() == "true"
OUTPUT_ROOT = Path(os.environ.get("SENSITIVITY_OUTPUT_ROOT", PROJECT_ROOT / "data/processed/legacy_sensitivity")).resolve()
FEATURES_PATH = Path(os.environ.get("SENSITIVITY_FEATURES_PATH", HISTORICAL_ROOT / "data/processed/learned_carbon/features_with_forest.parquet")).resolve()
REPORT_DIR = Path(os.environ.get("SENSITIVITY_REPORT_DIR", OUTPUT_ROOT / "reports" / f"robustness_{PROFILE}")).resolve()
SEEDS = (0, 1) if PROFILE == "test" else DEFAULT_SEEDS[PROFILE]
SCENARIOS = ("balanced",) if PROFILE == "test" else tuple(SCENARIO_LABELS)


In [2]:
if PROFILE == "test":
    position = np.linspace(0.0, 1.0, 12)
    context = pd.DataFrame({
        "cell_id": np.arange(1, 13), "forest_pct": 0.35 + 0.03 * position,
        "wetland_pct": 0.10 + 0.02 * position, "agriculture_pct": 0.30 - 0.03 * position,
        "grassland_pct": 0.15 - 0.02 * position, "urban_pct": np.full(12, 0.05),
        "water_pct": np.full(12, 0.05), "protected_overlap_pct": 0.05 * position,
        "wetland_suitability": 0.2 + 0.6 * position, "opportunity_cost_proxy": 0.1 + 0.5 * position,
        "predicted_tco2_ha_yr": 2.5 + 2.0 * position, "peat_overlap_pct": 0.4 * position,
    })
    feature_columns = ["wetland_suitability", "opportunity_cost_proxy"]
else:
    if not FEATURES_PATH.exists():
        raise FileNotFoundError(f"Missing historical feature input: {FEATURES_PATH}")
    context = pd.read_parquet(FEATURES_PATH)
    feature_columns = [name for name in ("urban_pct", "agriculture_pct", "grassland_pct", "forest_pct", "wetland_pct", "water_pct", "naturalness_score", "carbon_score", "protected_overlap_pct", "wetland_suitability", "biodiversity_proxy", "opportunity_cost_proxy", "rohemeeter_norm") if name in context]
    if not feature_columns:
        raise ValueError("No preserved Notebook 10 feature columns found")


In [3]:
expected_identity = current_artifact_identity(context, feature_columns, PROFILE)
inventory = inventory_artifacts(OUTPUT_ROOT, PROFILE, expected_identity=expected_identity)
manifest = build_biodiversity_manifest(profile=PROFILE, scenarios=SCENARIOS, seeds=SEEDS)
planned_total_runs = manifest_run_count(manifest)
manifest_summary(manifest)
display(manifest.head(12))
missing_manifest = missing_manifest_rows(manifest, inventory)
planned_new_runs = manifest_run_count(missing_manifest) if not missing_manifest.empty else 0
execution_manifest = full_manifest_for_partial_resume(manifest, missing_manifest)
print(f"Planned new biodiversity runs: {planned_new_runs}")
display(missing_manifest.head(12))


biodiversity: 72 optimizer runs (profile=full)


,experiment,sample_id,scenario,seed,profile,biodiversity_assumption,overrides,status
0,biodiversity,biodiversity__current,green_maximum,42,full,current,"{'scoring.biodiversity_value': [0.7, 0.9, 0.2,...",pending
1,biodiversity,biodiversity__current,green_maximum,73,full,current,"{'scoring.biodiversity_value': [0.7, 0.9, 0.2,...",pending
2,biodiversity,biodiversity__current,green_maximum,101,full,current,"{'scoring.biodiversity_value': [0.7, 0.9, 0.2,...",pending
3,biodiversity,biodiversity__current,food_security,42,full,current,"{'scoring.biodiversity_value': [0.7, 0.9, 0.2,...",pending
4,biodiversity,biodiversity__current,food_security,73,full,current,"{'scoring.biodiversity_value': [0.7, 0.9, 0.2,...",pending
5,biodiversity,biodiversity__current,food_security,101,full,current,"{'scoring.biodiversity_value': [0.7, 0.9, 0.2,...",pending
6,biodiversity,biodiversity__current,low_budget,42,full,current,"{'scoring.biodiversity_value': [0.7, 0.9, 0.2,...",pending
7,biodiversity,biodiversity__current,low_budget,73,full,current,"{'scoring.biodiversity_value': [0.7, 0.9, 0.2,...",pending
8,biodiversity,biodiversity__current,low_budget,101,full,current,"{'scoring.biodiversity_value': [0.7, 0.9, 0.2,...",pending
9,biodiversity,biodiversity__current,wetland_priority,42,full,current,"{'scoring.biodiversity_value': [0.7, 0.9, 0.2,...",pending


Planned new biodiversity runs: 36


,experiment,sample_id,scenario,seed,profile,biodiversity_assumption,overrides,status
0,biodiversity,biodiversity__open_habitat_focused,green_maximum,42,full,open_habitat_focused,"{'scoring.biodiversity_value': [0.6, 0.9, 0.2,...",pending
1,biodiversity,biodiversity__open_habitat_focused,green_maximum,73,full,open_habitat_focused,"{'scoring.biodiversity_value': [0.6, 0.9, 0.2,...",pending
2,biodiversity,biodiversity__open_habitat_focused,green_maximum,101,full,open_habitat_focused,"{'scoring.biodiversity_value': [0.6, 0.9, 0.2,...",pending
3,biodiversity,biodiversity__open_habitat_focused,food_security,42,full,open_habitat_focused,"{'scoring.biodiversity_value': [0.6, 0.9, 0.2,...",pending
4,biodiversity,biodiversity__open_habitat_focused,food_security,73,full,open_habitat_focused,"{'scoring.biodiversity_value': [0.6, 0.9, 0.2,...",pending
5,biodiversity,biodiversity__open_habitat_focused,food_security,101,full,open_habitat_focused,"{'scoring.biodiversity_value': [0.6, 0.9, 0.2,...",pending
6,biodiversity,biodiversity__open_habitat_focused,low_budget,42,full,open_habitat_focused,"{'scoring.biodiversity_value': [0.6, 0.9, 0.2,...",pending
7,biodiversity,biodiversity__open_habitat_focused,low_budget,73,full,open_habitat_focused,"{'scoring.biodiversity_value': [0.6, 0.9, 0.2,...",pending
8,biodiversity,biodiversity__open_habitat_focused,low_budget,101,full,open_habitat_focused,"{'scoring.biodiversity_value': [0.6, 0.9, 0.2,...",pending
9,biodiversity,biodiversity__open_habitat_focused,wetland_priority,42,full,open_habitat_focused,"{'scoring.biodiversity_value': [0.6, 0.9, 0.2,...",pending


In [4]:
if planned_new_runs:
    statuses = run_manifest(
        context, feature_columns, execution_manifest, OUTPUT_ROOT, PROFILE,
        overwrite=OVERWRITE, n_workers=min(N_WORKERS, planned_new_runs),
        progress=lambda completed, total, status: print(f"[{completed}/{total}] {status}"),
    )
    if statuses["status"].eq("failed").any():
        raise RuntimeError(statuses.loc[statuses["status"].eq("failed"), ["sample_id", "scenario", "seed", "error_message"]].to_string(index=False))
    display(statuses["status"].value_counts())
    print(f"Optimizer training executions: {statuses['status'].eq('completed').sum()}")
else:
    statuses = pd.DataFrame()
    print("All matching biodiversity artifacts already exist; nothing scheduled.")


[1/72] skipped
[2/72] skipped
[3/72] skipped
[4/72] skipped
[5/72] skipped
[6/72] skipped
[7/72] skipped
[8/72] skipped
[9/72] skipped
[10/72] skipped
[11/72] skipped
[12/72] skipped
[13/72] skipped
[14/72] skipped
[15/72] skipped
[16/72] skipped
[17/72] skipped
[18/72] skipped
[19/72] skipped
[20/72] skipped
[21/72] skipped
[22/72] skipped
[23/72] skipped
[24/72] skipped
[25/72] skipped
[26/72] skipped
[27/72] skipped
[28/72] skipped
[29/72] skipped
[30/72] skipped
[31/72] skipped
[32/72] skipped
[33/72] skipped
[34/72] skipped
[35/72] skipped
[36/72] skipped
[37/72] completed
[38/72] completed
[39/72] completed
[40/72] completed
[41/72] completed
[42/72] completed
[43/72] completed
[44/72] completed
[45/72] completed
[46/72] completed
[47/72] completed
[48/72] completed
[49/72] completed
[50/72] completed
[51/72] completed
[52/72] completed
[53/72] completed
[54/72] completed
[55/72] completed
[56/72] completed
[57/72] completed
[58/72] completed
[59/72] completed
[60/72] completed
[

status
skipped      36
completed    36
Name: count, dtype: int64

Optimizer training executions: 36


In [5]:
report_paths = build_robustness_report(
    OUTPUT_ROOT, REPORT_DIR, PROFILE, expected_identity=expected_identity,
)
display(pd.read_csv(report_paths["completeness"]))
display(pd.read_csv(report_paths["comparison_groups"]))
display(pd.read_csv(report_paths["rank_stability"]))
display(pd.read_csv(report_paths["parameter_importance"]))
display(pd.read_csv(report_paths["interactions"]))
display(pd.read_csv(report_paths["spatial_robustness"]).head())
conclusions = json.loads(report_paths["conclusions"].read_text(encoding="utf-8"))
display(pd.Series(conclusions, name="conclusion"))
print("Model and interaction conclusions are descriptive screening diagnostics, not statistical confirmation.")
print(f"Robustness report written to {REPORT_DIR}")


,experiment,profile,complete_runs,expected_runs,missing_runs,availability
0,baseline,full,18,18,0,complete
1,oat,full,108,108,0,complete
2,global,full,0,288,288,incomplete
3,interactions,full,150,150,0,complete
4,biodiversity,full,72,72,0,complete


,comparison_key,sample_id,seed,expected_scenarios,complete_scenarios,missing_scenarios,status
0,biodiversity__current|seed=42,biodiversity__current,42,"[""balanced"", ""food_security"", ""green_maximum"",...","[""balanced"", ""food_security"", ""green_maximum"",...",[],complete
1,biodiversity__current|seed=73,biodiversity__current,73,"[""balanced"", ""food_security"", ""green_maximum"",...","[""balanced"", ""food_security"", ""green_maximum"",...",[],complete
2,biodiversity__current|seed=101,biodiversity__current,101,"[""balanced"", ""food_security"", ""green_maximum"",...","[""balanced"", ""food_security"", ""green_maximum"",...",[],complete
3,biodiversity__forest_focused|seed=42,biodiversity__forest_focused,42,"[""balanced"", ""food_security"", ""green_maximum"",...","[""balanced"", ""food_security"", ""green_maximum"",...",[],complete
4,biodiversity__forest_focused|seed=73,biodiversity__forest_focused,73,"[""balanced"", ""food_security"", ""green_maximum"",...","[""balanced"", ""food_security"", ""green_maximum"",...",[],complete
5,biodiversity__forest_focused|seed=101,biodiversity__forest_focused,101,"[""balanced"", ""food_security"", ""green_maximum"",...","[""balanced"", ""food_security"", ""green_maximum"",...",[],complete
6,biodiversity__open_habitat_focused|seed=42,biodiversity__open_habitat_focused,42,"[""balanced"", ""food_security"", ""green_maximum"",...","[""balanced"", ""food_security"", ""green_maximum"",...",[],complete
7,biodiversity__open_habitat_focused|seed=73,biodiversity__open_habitat_focused,73,"[""balanced"", ""food_security"", ""green_maximum"",...","[""balanced"", ""food_security"", ""green_maximum"",...",[],complete
8,biodiversity__open_habitat_focused|seed=101,biodiversity__open_habitat_focused,101,"[""balanced"", ""food_security"", ""green_maximum"",...","[""balanced"", ""food_security"", ""green_maximum"",...",[],complete
9,biodiversity__lower_contrast|seed=42,biodiversity__lower_contrast,42,"[""balanced"", ""food_security"", ""green_maximum"",...","[""balanced"", ""food_security"", ""green_maximum"",...",[],complete


,scenario,first_place_frequency,median_rank,comparison_count,outcome
0,green_maximum,0.750000,1.0,12,biodiversity_gain
1,food_security,0.250000,4.0,12,biodiversity_gain
2,low_budget,0.000000,4.5,12,biodiversity_gain
3,wetland_priority,0.000000,2.0,12,biodiversity_gain
4,sustainable_agriculture,0.000000,6.0,12,biodiversity_gain
5,balanced,0.000000,4.0,12,biodiversity_gain
6,green_maximum,1.000000,1.0,12,carbon_gain
7,food_security,0.000000,3.0,12,carbon_gain
8,low_budget,0.000000,5.0,12,carbon_gain
9,wetland_priority,0.000000,2.0,12,carbon_gain


,parameter,spearman_rho,random_forest_importance,permutation_importance_mean,permutation_importance_sd,scenario,outcome,held_out_r2,min_held_out_r2,top_parameter_rank_consistent,top_parameter_repeat_dispersion_pass,model_fit_pass


,scenario,parameter_x,parameter_y,outcome,max_abs_interaction_residual,rms_interaction_residual,baseline_sd,max_abs_residual_to_noise,rms_residual_to_noise
0,balanced,scoring.agriculture_loss_cost,scoring.max_agriculture_loss_pct,biodiversity_gain,0.003011,0.001001,0.000783,3.844823,1.278485
1,balanced,scoring.agriculture_loss_cost,scoring.max_agriculture_loss_pct,carbon_gain,0.003454,0.001605,0.003084,1.119968,0.520298
2,balanced,scoring.agriculture_loss_cost,scoring.max_agriculture_loss_pct,cost,0.030483,0.013881,0.013346,2.284123,1.040129
3,balanced,scoring.agriculture_loss_cost,scoring.max_agriculture_loss_pct,changed_pct,0.008294,0.003781,0.007521,1.102767,0.502753
4,balanced,scoring.base_change_cost,max_changed_pct,biodiversity_gain,0.001421,0.000789,0.000783,1.814441,1.007498
5,balanced,scoring.base_change_cost,max_changed_pct,carbon_gain,0.002563,0.001239,0.003084,0.831255,0.401854
6,balanced,scoring.base_change_cost,max_changed_pct,cost,0.017534,0.008110,0.013346,1.313837,0.607701
7,balanced,scoring.base_change_cost,max_changed_pct,changed_pct,0.005237,0.002199,0.007521,0.696346,0.292442


,scenario,cell_id,modal_action,action_agreement,comparison_count,forest_target_mean,forest_target_sd,wetland_target_mean,wetland_target_sd,agriculture_target_mean,agriculture_target_sd,grassland_target_mean,grassland_target_sd
0,balanced,0,unchanged,1.0,12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,balanced,1,unchanged,1.0,12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,balanced,2,unchanged,1.0,12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,balanced,3,unchanged,1.0,12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,balanced,4,unchanged,1.0,12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


complete_comparison_group_count                                                   12
excluded_comparison_group_count                                                    0
excluded_comparison_group_keys                                                    []
excluded_incomplete_artifacts                                                      0
excluded_manifest_artifacts                                                      288
expected_comparison_group_count                                                   12
identity                           [full, a84b6ff57a18b16a679a406aed723b22d337fce...
interaction_criterion              {'interpretation': 'descriptive screening only...
interactions                                     screening-large-relative-to-seed-sd
missing_experiments                                                         [global]
parameter_importance                                                     unavailable
parameter_importance_criteria      {'descriptive_top_permutation_

Model and interaction conclusions are descriptive screening diagnostics, not statistical confirmation.
Robustness report written to C:\Users\risto\projects\et-landuse-neuroevolution\.worktrees\legacy-optimizer-sensitivity\data\processed\legacy_sensitivity\reports\robustness_full
